In [42]:
# Table 1 is a summary of all donors

import pandas as pd
import numpy as np

# Load the two metadata files
meta1 = pd.read_parquet('/home/adm808/New_CellMetadataSyn1848517.parquet')  # Mathys
meta2 = pd.read_parquet('/n/groups/patel/adithya/Validation_Metadata_Final.parquet')  # Lau

# Normalize key column names
meta1['dataset'] = 'Mathys'
meta2['dataset'] = 'Lau'

meta1['msex'] = meta1['msex'].replace({1.0: 1, 2.0: 0}).astype(int)
meta2['msex'] = meta2['SEX'].map({'M': 1, 'F': 0})
meta2['age_death'] = meta2['AGE']
meta2['dcfdx_lv'] = np.nan
meta2.loc[meta2['diagnosis'] == 1, 'dcfdx_lv'] = 4.0
meta2.loc[meta2['diagnosis'] == 0, 'dcfdx_lv'] = 1.0

# Harmonize braak stage
meta1['braak_stage'] = meta1['braaksc']
meta2['braak_stage'] = meta2['Braak tangle stage']

# Harmonize CERAD/age-related plaque score
meta1['ceradsc/age-related_plaque_score'] = meta1['ceradsc']
meta2['ceradsc/age-related_plaque_score'] = meta2['AGE-RELATED PLAQUE SCORE']

# Merge APOE genotype columns
numeric_to_string = {
    22.0: 'E2/E2',
    23.0: 'E2/E3',
    24.0: 'E2/E4',
    33.0: 'E3/E3',
    34.0: 'E3/E4',
    44.0: 'E4/E4'
}
meta1['apoe_genotype'] = pd.to_numeric(meta1['apoe_genotype'], errors='coerce')
meta2['apoe_genotype'] = pd.to_numeric(meta2['apoe_genotype'], errors='coerce')
meta1['apoe_genotype'] = meta1['apoe_genotype'].map(numeric_to_string)
meta2['apoe_genotype'] = meta2['apoe_genotype'].map(numeric_to_string)

# Include only Mathys education and PMI; leave as Na for Lau
meta2['educ'] = np.nan
meta1['pmi'] = meta1['pmi']
meta2['pmi'] = meta2['pmi']

# Concatenate metadata
metadata = pd.concat([meta1, meta2], axis=0, ignore_index=True)

# Filter down to only cells with valid sample and broad.cell.type
metadata = metadata.dropna(subset=['sample', 'broad.cell.type'])

# Keep only relevant cell types
cell_types = ['Mic', 'Ast', 'Opc', 'In', 'Ex', 'Oli']
metadata = metadata[metadata['broad.cell.type'].isin(cell_types)]

# Calculate percentage of each of the 6 cell types per sample
cell_counts = metadata.groupby(['sample', 'broad.cell.type']).size().unstack(fill_value=0)
cell_percents = cell_counts.div(cell_counts.sum(axis=1), axis=0) * 100
cell_percents = cell_percents[cell_types]  # Ensure consistent column order

# Merge back with sample-level metadata
sample_cols = ['sample', 'age_death', 'msex', 'apoe_genotype', 'educ', 'pmi', 'dataset', 'ceradsc/age-related_plaque_score', 'dcfdx_lv', 'braak_stage', 'projid']
sample_metadata = metadata.drop_duplicates(subset=['sample'])[sample_cols].set_index('sample')
merged = sample_metadata.join(cell_percents, how='left')


# Add diagnosis column: AD if dcfdx_lv is 4 or 5, else Control
# Reset index first so 'sample' is a column again
merged = merged.reset_index()
merged['cell_counts'] = merged['sample'].map(metadata['sample'].value_counts())
dcfdx_map = metadata.drop_duplicates(subset='sample').set_index('sample')['dcfdx_lv']
merged['dcfdx_lv'] = merged['sample'].map(dcfdx_map)
merged['diagnosis'] = np.where(merged['dcfdx_lv'].isin([4.0, 5.0]), 'AD', 'Control')

# Drop any duplicate sample rows (keep first)
merged = merged[~merged.index.duplicated(keep='first')]
merged = merged.rename(columns={ct: f"{ct} (%)" for ct in cell_types})



# Final result
merged = merged.reset_index()
merged = merged.drop(columns='index', errors='ignore')
merged.columns.name = None
merged.head()
merged.loc[merged['dataset'] == 'Lau', 'dcfdx_lv'] = np.nan
merged.to_csv("/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Data/Full_Experiment_Metadata.csv", index=False)

In [43]:
merged.columns

Index(['sample', 'age_death', 'msex', 'apoe_genotype', 'educ', 'pmi',
       'dataset', 'ceradsc/age-related_plaque_score', 'dcfdx_lv',
       'braak_stage', 'projid', 'Mic (%)', 'Ast (%)', 'Opc (%)', 'In (%)',
       'Ex (%)', 'Oli (%)', 'cell_counts', 'diagnosis'],
      dtype='object')

In [44]:
merged

# Convert msex from 1/0 to Male/Female
merged['Sex'] = merged['msex'].map({1: 'Male', 0: 'Female'})

# Drop the old msex column
merged = merged.drop(columns='msex')

# Rename specific columns
merged = merged.rename(columns={
    'educ': 'Education',
    'apoe_genotype': 'APOE Genotype',
    'cell_counts': 'Cell Count',
    'age_death': 'Age at Death'
})

# Capitalize first letter of all other columns
merged.columns = [col[0].upper() + col[1:] if not col.istitle() else col for col in merged.columns]

# Ensure column order: Sex right after Age at Death
cols = list(merged.columns)
if 'Sex' in cols and 'Age at Death' in cols:
    cols.insert(cols.index('Age at Death') + 1, cols.pop(cols.index('Sex')))
merged = merged[cols]

In [60]:
merged

merged.to_csv("/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Data/Full_Experiment_Metadata.csv", index=False)

In [61]:
merged

,Sample,Age at Death,Sex,APOE Genotype,Education,PMI,Dataset,CERAD/AGE RELATED PLAQUE SCORE,DCFDX LV,Braak Stage,ProjectID,Mic (%),Ast (%),Opc (%),In (%),Ex (%),Oli (%),Cell Count,Diagnosis
0,D1,80.098562628336751,Male,E3/E3,22.0,1.333333,Mathys,4.0,2.0,3.0,11409232,4.356846,1.037344,1.659751,11.410788,64.730290,16.804979,482,Control
1,D2,89.026694045174537,Male,E3/E4,22.0,3.500000,Mathys,1.0,4.0,6.0,11336574,11.262799,5.460751,2.730375,7.849829,40.955631,31.740614,293,AD
2,D3,88.399726214921287,Male,E3/E4,18.0,8.583333,Mathys,4.0,1.0,3.0,10260309,8.854167,7.986111,3.732639,18.055556,36.024306,25.347222,1152,Control
3,D4,88.468172484599592,Male,E4/E4,19.0,17.916667,Mathys,2.0,4.0,3.0,10248033,2.324431,6.775470,6.577646,16.369931,41.889219,26.063304,2022,AD
4,D5,90+,Female,E2/E3,23.0,4.166667,Mathys,4.0,1.0,1.0,20207013,3.621939,5.450155,5.139703,15.281131,43.739220,26.767851,2899,Control
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64,D65,74,Female,E3/E3,NaN,39.500000,Lau,0,NaN,1.0,NC14,0.265932,18.529775,0.000000,29.917371,35.729889,15.557033,10529,Control
65,D66,79,Female,E3/E3,NaN,48.000000,Lau,0,NaN,2.0,NC15,4.199900,3.751246,0.000000,28.913260,58.437188,4.698405,8024,Control
66,D67,89,Male,E3/E4,NaN,54.500000,Lau,0,NaN,2.0,NC16,1.575737,18.146391,0.000000,15.723484,49.508641,15.045747,5902,Control
67,D68,78,Male,E3/E3,NaN,51.500000,Lau,0,NaN,2.0,NC17,5.610657,14.413958,0.000000,14.891769,38.919858,26.163759,13813,Control


In [46]:
# Ensure projid column is string type first
merged['Projid'] = merged['Projid'].apply(
    lambda x: str(int(x)) if pd.notna(x) and str(x).endswith('.0') else str(x) if pd.notna(x) else np.nan
)

# Fill Lau projid with their sample IDs
mask_lau = merged['Dataset'] == 'Lau'
merged.loc[mask_lau, 'Projid'] = merged.loc[mask_lau, 'Sample'].astype(str)

In [47]:
merged
merged['Sample'] = [f"D{i}" for i in range(1, len(merged) + 1)]

In [58]:
merged = merged.rename(columns={
    'Pmi': 'PMI',
    'Ceradsc/age-related_plaque_score': 'CERAD/AGE RELATED PLAQUE SCORE',
    'Dcfdx_lv': 'DCFDX LV',
    'Braak_stage': 'Braak Stage',
    'Projid': 'ProjectID'
})

In [62]:
merged.columns

Index(['Sample', 'Age at Death', 'Sex', 'APOE Genotype', 'Education', 'PMI',
       'Dataset', 'CERAD/AGE RELATED PLAQUE SCORE', 'DCFDX LV', 'Braak Stage',
       'ProjectID', 'Mic (%)', 'Ast (%)', 'Opc (%)', 'In (%)', 'Ex (%)',
       'Oli (%)', 'Cell Count', 'Diagnosis'],
      dtype='object')

In [77]:
import pandas as pd
import numpy as np

# Split groups
ad_df   = merged[merged['Diagnosis'] == 'AD']
ctrl_df = merged[merged['Diagnosis'] == 'Control']

# ---------- helpers ----------
def mean_sd(series):
    s = pd.to_numeric(series, errors='coerce').dropna()
    return "" if s.empty else f"{s.mean():.2f} ± {s.std():.2f}"

def count_in_group(df, col, value):
    s = df[col].dropna()
    return int((s == value).sum())

def list_present(series):
    """List distinct non-NaN values; normalize '3.0'->'3'; mixed-type safe sort."""
    s = series.dropna().astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
    vals = [v for v in s.unique().tolist() if str(v).lower() != 'nan']
    def key(x):
        try:
            return (0, float(x))
        except:
            return (1, x)
    return ", ".join(sorted(vals, key=key))

def sex_n_pct(df, sex_label):
    s = df['Sex'].dropna()
    n = len(s)
    if n == 0:
        return ""
    k = (s == sex_label).sum()
    return f"{k} ({100*k/n:.1f}%)"

# ---------- build table rows ----------
rows = []

# Continuous: mean ± SD
for col in ['Age at Death', 'Education', 'Cell Count',
            'Mic (%)', 'Ast (%)', 'Opc (%)', 'In (%)', 'Ex (%)', 'Oli (%)']:
    rows.append({
        "Summary": col,
        "Control": mean_sd(ctrl_df[col]),
        "Alzheimer's": mean_sd(ad_df[col])
    })

# Sex (n, %) rows
rows.append({
    "Summary": "Male sex, n (%)",
    "Control": sex_n_pct(ctrl_df, "Male"),
    "Alzheimer's": sex_n_pct(ad_df, "Male")
})
rows.append({
    "Summary": "Female sex, n (%)",
    "Control": sex_n_pct(ctrl_df, "Female"),
    "Alzheimer's": sex_n_pct(ad_df, "Female")
})

# APOE genotype: one row per genotype, counts only
apoe_levels = ['E2/E3','E2/E4','E3/E3','E3/E4','E4/E4']
for g in apoe_levels:
    rows.append({
        "Summary": f"APOE {g}, n",
        "Control": count_in_group(ctrl_df, 'APOE Genotype', g),
        "Alzheimer's": count_in_group(ad_df, 'APOE Genotype', g)
    })

# Pathology/staging: list all distinct values present (exclude NaN)
for col in ['CERAD/AGE RELATED PLAQUE SCORE', 'DCFDX LV', 'Braak Stage']:
    rows.append({
        "Summary": col,
        "Control": list_present(ctrl_df[col]),
        "Alzheimer's": list_present(ad_df[col])
    })

table1_summary = pd.DataFrame(rows, columns=["Summary", "Control", "Alzheimer's"])

# Save
out_csv = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Data/Table1_donor_summary_main.csv"
table1_summary.to_csv(out_csv, index=False)

In [78]:
table1_summary

,Summary,Control,Alzheimer's
0,Age at Death,83.99 ± 5.11,80.68 ± 9.33
1,Education,18.42 ± 3.49,19.36 ± 2.65
2,Cell Count,3404.14 ± 3850.68,3638.68 ± 4180.24
3,Mic (%),3.30 ± 2.65,3.92 ± 2.94
4,Ast (%),6.38 ± 5.06,7.59 ± 4.86
5,Opc (%),2.44 ± 1.88,2.55 ± 2.24
6,In (%),15.74 ± 6.75,15.49 ± 5.35
7,Ex (%),47.48 ± 13.94,46.00 ± 12.36
8,Oli (%),24.67 ± 11.56,24.44 ± 12.20
9,"Male sex, n (%)",21 (60.0%),17 (50.0%)


In [13]:
meta.columns

Index(['sample', 'age_death', 'msex', 'apoe_genotype', 'educ', 'pmi',
       'dataset', 'ceradsc/age-related_plaque_score', 'dcfdx_lv',
       'braak_stage', 'Mic (%)', 'Ast (%)', 'Opc (%)', 'In (%)', 'Ex (%)',
       'Oli (%)', 'cell_counts', 'diagnosis'],
      dtype='object')

In [14]:
meta['diagnosis'].value_counts()

diagnosis
Control    35
AD         34
Name: count, dtype: int64

In [ ]:
,e

In [6]:
meta2 = pd.read_parquet('/n/groups/patel/adithya/Validation_Metadata_Final.parquet')

In [ ]:
import numpy as np
meta2['dcfdx_lv'] = np.nan
meta2.loc[meta2['diagnosis'] == 1, 'dcfdx_lv'] = 4.0
meta2.loc[meta2['diagnosis'] == 0, 'dcfdx_lv'] = 1.0
meta2

,TAG,sample,diagnosis,ID,CONDITION,AGE,SEX,PMD,HIST DIAGNOSIS,DIAG 1,...,pmi,educ,cts_mmse30_lv,age_death,apoe_genotype,alzheimers_or_control,msex,pmi_bin,predicted_subcluster,dcfdx_lv
0,AAACCCAAGCTGAAAT-1,AD1,1,AD1,AD,69,M,7.75,AD definite,AD,...,7.75,16,25,69,33.0,1,1,0,Ex2,4.0
1,AAACCCACAAATGGTA-1,AD1,1,AD1,AD,69,M,7.75,AD definite,AD,...,7.75,16,25,69,33.0,1,1,0,Mic0,4.0
2,AAACCCACAATGAAAC-1,AD1,1,AD1,AD,69,M,7.75,AD definite,AD,...,7.75,16,25,69,33.0,1,1,0,Oli1,4.0
3,AAACCCAGTAACCCTA-1,AD1,1,AD1,AD,69,M,7.75,AD definite,AD,...,7.75,16,25,69,33.0,1,1,0,Ex8,4.0
4,AAACCCAGTAGCGAGT-1,AD1,1,AD1,AD,69,M,7.75,AD definite,AD,...,7.75,16,25,69,33.0,1,1,0,Oli1,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
172509,TTTGTTGTCCGAGCTG-1,NC18,0,NC18,NC,94,F,29.50,"Control, moderate CAA",CONTROL,...,29.50,16,25,94,34.0,0,0,1,Ex2,1.0
172510,TTTGTTGTCCTAACAG-1,NC18,0,NC18,NC,94,F,29.50,"Control, moderate CAA",CONTROL,...,29.50,16,25,94,34.0,0,0,1,Ex2,1.0
172511,TTTGTTGTCCTCACGT-1,NC18,0,NC18,NC,94,F,29.50,"Control, moderate CAA",CONTROL,...,29.50,16,25,94,34.0,0,0,1,Mic0,1.0
172512,TTTGTTGTCGGCATTA-1,NC18,0,NC18,NC,94,F,29.50,"Control, moderate CAA",CONTROL,...,29.50,16,25,94,34.0,0,0,1,Ex2,1.0


In [2]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

cell_types = ['Mic', 'Ast', 'In', 'Ex', 'Opc', 'Oli']

MODELS = [
    {"label": "Combined (Clinical)", "template": "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_both_new/{cell_type}"},
    {"label": "Genes (Clinical)", "template": "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new/{cell_type}"},
    {"label": "Demo + APOE (Clinical)", "template": "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_demo_new/{cell_type}"},
    {"label": "APOE + Genes (Clinical)", "template": "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_apoe_new/{cell_type}"},

    {"label": "Combined (Pathological)", "template": "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_both_cerad/{cell_type}"},
    {"label": "Genes (Pathological)", "template": "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_cerad/{cell_type}"},
    {"label": "Demo + APOE (Pathological)", "template": "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_demo_cerad/{cell_type}"},
    {"label": "APOE + Genes (Pathological)", "template": "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_apoe_cerad/{cell_type}"},

    {"label": "Genes (Validation)", "template": "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Validation_Multirun_cell_on_cell_genes_Lau_rfe/{cell_type}"},
    {"label": "APOE + Genes (Validation)", "template": "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Validation_Multirun_cell_on_cell_genes_APOE_genes/{cell_type}"},
]

output_excel = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Data/Auc_summary_table.xlsx"
writer = pd.ExcelWriter(output_excel, engine='xlsxwriter')

for cell_type in cell_types:
    rows = []
    for model in MODELS:
        aucs = []
        for split in range(1, 6):
            pred_file = os.path.join(model["template"].format(cell_type=cell_type),
                                     f"split_{split}", "test_predictions.csv")
            if not os.path.exists(pred_file):
                aucs.append(np.nan)
                continue
            df = pd.read_csv(pred_file)
            y_true = df['true_label']
            y_score = df['predicted_proba']
            auc_val = roc_auc_score(y_true, y_score)
            aucs.append(auc_val)

        mean_auc = np.nanmean(aucs)
        std_auc = np.nanstd(aucs)
        rows.append([model["label"]] + aucs + [mean_auc, std_auc])

    cols = ["Model"] + [f"Split_{i}" for i in range(1, 6)] + ["Mean_AUC", "Std_AUC"]
    df_ct = pd.DataFrame(rows, columns=cols)
    df_ct.to_excel(writer, sheet_name=cell_type, index=False)

writer.close()
print(f"Saved AUC summary to {output_excel}")

/tmp/ipykernel_2925461/179949101.py:42: RuntimeWarning: Mean of empty slice
  mean_auc = np.nanmean(aucs)
/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


Saved AUC summary to /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Data/Auc_summary_table.xlsx
